# 01 Validation: OpenICU → YAIB dynamic table

Use this notebook after `01_openicu_to_yaib_dyn.ipynb`.

In [ ]:
from pathlib import Path
import polars as pl

from openicu_yaib.concepts import DYNAMIC_VARS
from openicu_yaib.ricu_meta import RicuConceptMeta
from openicu_yaib.validation import (
    scan_dynamic,
    table_summary,
    validate_dynamic_columns,
    concept_coverage,
    ricu_range_report,
    stay_windows_from_dyn,
    compare_dyn_to_icustays,
    debug_concept_against_output,
)

In [ ]:
CONCEPT_ROOT = Path("~/OpenICU.example/output/project/workspace/concept")
ICUSTAYS_CSV = Path("~/physionet.org/files/mimiciv/3.1/icu/icustays.csv.gz")
RICU_CONCEPT_DICT = Path("~/ricu/inst/extdata/config/concept-dict.json")
OUTPUT = Path("~/output/openicu_dyn.parquet")

DATASET = "mimic-iv"
VERSION = "1.0.0"

df = scan_dynamic(OUTPUT)
ricu_meta = RicuConceptMeta.from_json(RICU_CONCEPT_DICT)

## Table summary

In [ ]:
table_summary(df)

## Column completeness

In [ ]:
column_report = validate_dynamic_columns(df, DYNAMIC_VARS)
column_report

In [ ]:
assert not column_report["missing_cols"], f"Missing expected columns: {column_report['missing_cols']}"

## Coverage

In [ ]:
coverage = concept_coverage(df, DYNAMIC_VARS)
coverage

In [ ]:
coverage.filter(pl.col("concept").is_in([
    "hr", "resp", "sbp", "dbp", "map", "temp", "o2sat", "glu", "crea", "wbc"
]))

## RICU range report

In [ ]:
range_report = ricu_range_report(df, ricu_meta, DYNAMIC_VARS)
range_report

In [ ]:
range_report.filter(pl.col("below_ricu_min") | pl.col("above_ricu_max"))

## Stay-window summary from final dynamic table

In [ ]:
dyn_windows = stay_windows_from_dyn(df)
dyn_windows.head()

In [ ]:
dyn_windows.select([
    pl.len().alias("n_stays"),
    pl.col("dyn_start").min().alias("min_dyn_start"),
    pl.col("dyn_end").max().alias("max_dyn_end"),
    pl.col("missing_grid_points").min().alias("min_missing_grid_points"),
    pl.col("missing_grid_points").max().alias("max_missing_grid_points"),
])

## Compare final dyn windows against MIMIC icustays-derived windows

In [ ]:
window_comparison = compare_dyn_to_icustays(df, ICUSTAYS_CSV, end_rounding="floor")
window_comparison.head()

In [ ]:
window_comparison.select([
    pl.len().alias("n_stays"),
    pl.col("diff_end").min().alias("min_diff_end"),
    pl.col("diff_end").max().alias("max_diff_end"),
    (pl.col("diff_end") == 0).sum().alias("n_same_end"),
    (pl.col("diff_end") < 0).sum().alias("n_dyn_shorter"),
    (pl.col("diff_end") > 0).sum().alias("n_dyn_longer"),
])

## Debug selected concepts against final output

In [ ]:
debug_concept_against_output(
    concept="hr",
    output_dyn=OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_meta=ricu_meta,
    dataset=DATASET,
    version=VERSION,
    aggregate="mean",
)

In [ ]:
debug_concept_against_output(
    concept="crea",
    output_dyn=OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_meta=ricu_meta,
    dataset=DATASET,
    version=VERSION,
    aggregate="mean",
)

In [ ]:
debug_concept_against_output(
    concept="urine",
    output_dyn=OUTPUT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_meta=ricu_meta,
    dataset=DATASET,
    version=VERSION,
    aggregate="mean",
)